In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(device)

In [ ]:
# 10. 高频复杂度速查（元素数，不计 batch/layer/dtype）
def kv_cache_elements(seq_len, head_dim, num_heads, num_kv_heads=None, kv_lora_rank=None):
    if kv_lora_rank is not None:
        return seq_len * kv_lora_rank                 # 简化 MLA latent cache
    num_kv_heads = num_heads if num_kv_heads is None else num_kv_heads
    return 2 * seq_len * num_kv_heads * head_dim      # K 和 V

T, H, Hkv, Dh, R = 4096, 32, 8, 128, 512
print({
    "MHA": kv_cache_elements(T, Dh, H),
    "GQA": kv_cache_elements(T, Dh, H, num_kv_heads=Hkv),
    "MQA": kv_cache_elements(T, Dh, H, num_kv_heads=1),
    "简化MLA": kv_cache_elements(T, Dh, H, kv_lora_rank=R),
})